Fast/Slow Moving Analysis
Measures how quickly each item moves from stock arrival (Purchase) to sale, using lot-level matching. Precomputes a percentile rank so any threshold can be applied instantly.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.item_velocity import compute_item_velocity, filter_by_percentile
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/analysis/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])
print(f"Analysis silver: {analysis_silver.shape}")

Compute item velocity

In [0]:
item_velocity = compute_item_velocity(analysis_silver)
print(item_velocity.shape)
print(item_velocity.head(10))

Save

In [0]:
item_names = analysis_silver[["item_no", "item_description"]].drop_duplicates(subset="item_no")

item_classification = item_classification.merge(item_names, on="item_no", how="left")
item_velocity = item_velocity.merge(item_names, on="item_no", how="left")

In [0]:
save_gold(blob_service, item_velocity, f"{ANALYSIS_BASE}/analysis/item_velocity.parquet")

buffer = io.BytesIO()
item_velocity.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/analysis/item_velocity.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved item_velocity.parquet and .xlsx")

Demo: threshold as pure filter

In [0]:
dbutils.widgets.text("velocity_percentile_threshold", "70")
velocity_threshold = float(dbutils.widgets.get("velocity_percentile_threshold"))

fast_moving = filter_by_percentile(item_velocity, "velocity", velocity_threshold)
slow_moving = item_velocity[item_velocity["velocity_percentile"] < (100 - velocity_threshold)]

print(f"Fast moving items (>= {velocity_threshold}th percentile): {len(fast_moving)}")
print(fast_moving[["item_no", "avg_days_to_sell", "matched_events"]].head(15))

print(f"\nSlow moving items: {len(slow_moving)}")
print(slow_moving[["item_no", "avg_days_to_sell", "matched_events"]].head(15))